# 01 — Frame Selection

Extract frames from raw video, run pretrained ViTPose++ to identify
low-confidence frames (for active-learning annotation priority).

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

## 1. Extract frames from video

Run the extraction script (or call it here):

In [ ]:
# Option A: run the script from the notebook
# !python ../scripts/extract_frames.py --input ../data/raw_video/ --output ../data/frames/ --fps 5 --blur-threshold 100

# Option B: call the function directly
from walker_gait.pipeline import WalkerGaitPipeline

frames_dir = Path("../data/frames")
frame_paths = sorted(frames_dir.glob("*.png"))
print(f"Found {len(frame_paths)} extracted frames")

## 2. Run pretrained model to find low-confidence frames

In [ ]:
# Initialize pipeline with pretrained weights (no fine-tuning yet)
pipeline = WalkerGaitPipeline(
    pose_model_name="usyd-community/vitpose-plus-large",
    device="cuda",
)

In [ ]:
# Score each frame by mean keypoint confidence
frame_scores = []
for p in tqdm(frame_paths[:100], desc="Scoring frames"):  # adjust range
    image = Image.open(p).convert("RGB")
    result = pipeline(image)
    if len(result.scores) > 0:
        mean_score = result.scores[0].mean()
    else:
        mean_score = 0.0
    frame_scores.append((p.name, float(mean_score)))

# Sort by confidence — annotate low-confidence frames first
frame_scores.sort(key=lambda x: x[1])
print("\nLowest confidence frames (annotate these first):")
for name, score in frame_scores[:20]:
    print(f"  {name}: {score:.3f}")

## 3. Visualize a sample

In [ ]:
from walker_gait.utils import draw_keypoints_on_image

# Pick a low-confidence frame to inspect
sample_path = frames_dir / frame_scores[0][0]
image = Image.open(sample_path).convert("RGB")
result = pipeline(image)

if len(result.keypoints) > 0:
    vis = draw_keypoints_on_image(
        np.array(image), result.keypoints, result.scores
    )
    plt.figure(figsize=(10, 8))
    plt.imshow(vis)
    plt.title(f"{sample_path.name} — mean score: {frame_scores[0][1]:.3f}")
    plt.axis("off")
    plt.show()